# M2 sales-share target audit v0.1

This notebook validates the tracked aggregate evidence for the 2026-07-25 target migration. It reads no private rows, provider, database, or holdout.

In [1]:
from pathlib import Path
import json

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "package.json").is_file())
candidate_path = root / "docs/analysis/m2-current/M2-current-sales-share-candidate-v0.6.json"
previous_path = root / "docs/analysis/m2-current/M2-current-multi-resolution-candidate-v0.5.json"
candidate = json.loads(candidate_path.read_text(encoding="utf-8"))
previous = json.loads(previous_path.read_text(encoding="utf-8"))


In [2]:
frozen = candidate["targetMigration"]["frozenTargetIsolation"]
dense = candidate["targetMigration"]["denseTargetIsolation"]
summary = {
    "target": candidate["target"],
    "frozenCaseCount": frozen["caseCount"],
    "frozenTargetChangedCaseCount": frozen["targetChangedCaseCount"],
    "denseCaseCount": dense["caseCount"],
    "denseTargetChangedCaseCount": dense["targetChangedCaseCount"],
    "frozenIsolatedBuyoutCaseSum": frozen["isolatedBuyoutCashCaseSum"],
    "denseIsolatedBuyoutCaseSum": dense["isolatedBuyoutCashCaseSum"],
    "frozenClassificationUncertainShare": frozen["classificationUncertainCashShare"],
    "denseClassificationUncertainShare": dense["classificationUncertainCashShare"],
    "workWape": candidate["pointComparisonToPrevious"]["comparison"]["candidate"]["wape"],
    "portfolioWape": candidate["multiResolution"]["portfolioReconstruction"]["candidate"]["overall"]["wape"],
    "previousPortfolioWape": previous["multiResolution"]["portfolioReconstruction"]["candidate"]["overall"]["wape"],
}
summary

{'target': 'future_sales_share_cash',
 'frozenCaseCount': 7851,
 'frozenTargetChangedCaseCount': 0,
 'denseCaseCount': 56856,
 'denseTargetChangedCaseCount': 0,
 'frozenIsolatedBuyoutCaseSum': 4800850.153400008,
 'denseIsolatedBuyoutCaseSum': 11578794.999800006,
 'frozenClassificationUncertainShare': 2.7284304596833668e-06,
 'denseClassificationUncertainShare': 2.865143615676321e-06,
 'workWape': 0.50557140186362,
 'portfolioWape': 0.11681933893326385,
 'previousPortfolioWape': 0.11681933893326385}

In [3]:
assert summary["target"] == "future_sales_share_cash"
assert summary["frozenCaseCount"] == 7851
assert summary["denseCaseCount"] == 56856
assert summary["frozenTargetChangedCaseCount"] == 0
assert summary["denseTargetChangedCaseCount"] == 0
assert abs(summary["portfolioWape"] - summary["previousPortfolioWape"]) < 1e-15
assert frozen["maximumAbsoluteConservationDifference"] <= 0.000001
assert dense["maximumAbsoluteConservationDifference"] <= 0.000001
assert candidate["acceptance"]["allBuyoutExcludedFromTrainingLabels"] is True
assert candidate["acceptance"]["targetClassificationPassed"] is False
"All tracked aggregate target-migration assertions passed."


'All tracked aggregate target-migration assertions passed.'

## Interpretation

The migration changes the current and future target contract but not the numeric labels in the frozen authority. It therefore cannot explain or repair the current work-level error. The nonzero classification-uncertainty share remains a strict data-quality blocker.